# EXO-INSIGHT — Milestone 4: Scientific Validation & Potentially Temperate Screening

**Goal:** extend Milestone 3 with scientific-data review, uncertainty and mass-provenance checks, unusual-value flags, and a transparent evidence table for potentially temperate candidates.

**Scientific safeguards**
- This is not a habitability determination.
- `pl_eqt` (equilibrium temperature) and `pl_insol` (incident stellar flux) are complementary, not interchangeable.
- No arbitrary temperature/insolation cut-offs are assumed. Enter thresholds only when supported by an approved research source or supervisor protocol.
- Missing values remain missing; no global imputation or row deletion.
- IQR outlier flags mean “review this value,” not “this value is wrong.”

Expected files:
- `DATA/processed/exo_insight_preprocessed.csv`
- `DATA/raw/PSCompPars.csv`

The notebook writes new reports to `DATA/processed/milestone4/` and does not overwrite the raw or Milestone 3 files.

## 1. Imports

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
warnings.filterwarnings("ignore", category=RuntimeWarning)
print("pandas:", pd.__version__, "| numpy:", np.__version__)

## 2. Locate and load project data

The notebook checks the current Jupyter folder and its parent folders first, then falls back to the known Windows project directory. If your project is stored elsewhere, change the fallback path in the next cell.


In [ ]:
# Locate the project folder. First try the current Jupyter folder and its parents;
# if that fails, use the known Windows project location.
PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    *Path.cwd().parents,
    Path(r"S:\Aivancity\PGE 3\Coding for AI\EXO INSIGHTS"),
]

def locate_root(candidates):
    checked = []
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate in checked:
            continue
        checked.append(candidate)
        m3_candidate = candidate / "DATA" / "processed" / "exo_insight_preprocessed.csv"
        raw_candidate = candidate / "DATA" / "raw" / "PSCompPars.csv"
        if m3_candidate.is_file() and raw_candidate.is_file():
            return candidate

    checked_paths = "\n".join(f" - {p}" for p in checked)
    raise FileNotFoundError(
        "Could not locate both required EXO-INSIGHT input files.\n"
        "Update PROJECT_ROOT below to the folder containing the DATA directory.\n"
        f"Folders checked:\n{checked_paths}"
    )

PROJECT_ROOT = locate_root(PROJECT_ROOT_CANDIDATES)
M3_PATH = PROJECT_ROOT / "DATA" / "processed" / "exo_insight_preprocessed.csv"
RAW_PATH = PROJECT_ROOT / "DATA" / "raw" / "PSCompPars.csv"

print("Project root:", PROJECT_ROOT)
print("Milestone 3 dataset:", M3_PATH)
print("NASA raw dataset:", RAW_PATH)
print("Both input files found successfully.")

m3 = pd.read_csv(M3_PATH)
raw = pd.read_csv(RAW_PATH, low_memory=False, comment="#")
print("M3:", m3.shape, "| NASA raw:", raw.shape)
display(m3.head())


## 3. Milestone 3 handover validation

In [ ]:
# Ensure the Milestone 3 dataset is loaded before validating its columns.
# This guard also makes the cell recoverable if the Jupyter kernel was restarted
# or the earlier data-loading cell was skipped.
from pathlib import Path
import pandas as pd

if "M3_PATH" not in globals():
    project_root = Path(r"S:\Aivancity\PGE 3\Coding for AI\EXO INSIGHTS")
    M3_PATH = project_root / "DATA" / "processed" / "exo_insight_preprocessed.csv"

if "m3" not in globals():
    if not Path(M3_PATH).is_file():
        raise FileNotFoundError(
            f"Milestone 3 dataset not found: {M3_PATH}. "
            "Run the project-path setup cell first or correct M3_PATH."
        )
    m3 = pd.read_csv(M3_PATH, low_memory=False)
    print("Loaded Milestone 3 dataset:", m3.shape)

required = [
    "pl_name", "hostname", "discoverymethod", "disc_year", "pl_rade",
    "pl_bmasse", "pl_eqt", "pl_insol", "pl_orbper", "pl_orbsmax",
    "st_teff", "st_rad", "st_mass", "sy_dist", "pl_bmassprov"
]
missing_cols = [c for c in required if c not in m3.columns]
if missing_cols:
    raise ValueError(f"Missing expected M3 columns: {missing_cols}")

handover = pd.DataFrame([{
    "rows": len(m3), "columns": len(m3.columns),
    "duplicate_rows": int(m3.duplicated().sum()),
    "duplicate_planet_names": int(m3["pl_name"].duplicated().sum()),
    "missing_required_columns": ", ".join(missing_cols) or "None"
}])
display(handover)
assert m3.duplicated().sum() == 0
assert not m3["pl_name"].duplicated().any()
print("Handover checks passed.")


## 4. Recover uncertainty columns from the original NASA file
Milestone 3 retained 15 research columns. This section checks the original source for associated uncertainty fields, then joins them by planet name. It does not assume every uncertainty field exists.

In [ ]:
# NASA Exoplanet Archive CSV exports include metadata lines beginning with '#'.
# The raw file is therefore loaded with comment="#" in Section 2.
measurement_tokens = (
    "pl_eqt", "pl_insol", "pl_bmasse", "pl_rade", "pl_orbper",
    "pl_orbsmax", "st_teff", "st_rad", "st_mass"
)
uncertainty_cols = [
    c for c in raw.columns
    if c.lower().endswith(("err1", "err2"))
    and any(token in c.lower() for token in measurement_tokens)
]

# Retain a small number of source metadata fields needed to interpret the
# habitable-zone model's applicability and zero-insolation records.
source_metadata_cols = [
    c for c in ["st_spectype", "pl_insollim", "st_tefflim"]
    if c in raw.columns
]
source_join_cols = ["pl_name"] + uncertainty_cols + source_metadata_cols

if raw["pl_name"].duplicated().any():
    raise ValueError("Raw file contains repeated planet names; a one-to-one join is unsafe.")

analysis = m3.merge(
    raw[source_join_cols],
    on="pl_name", how="left", validate="one_to_one"
)
print("Uncertainty fields found:", uncertainty_cols)
print("Source metadata recovered:", source_metadata_cols)
print("Joined dataset:", analysis.shape)
display(analysis[["pl_name"] + uncertainty_cols + source_metadata_cols].head())


## 5. Physical-value review and zero-insolation investigation

In [ ]:
physical_fields = [
    c for c in ["pl_eqt", "pl_insol", "pl_orbsmax", "pl_orbper",
                "pl_rade", "pl_bmasse", "st_teff", "st_rad", "st_mass"]
    if c in analysis.columns
]
rows = []
for col in physical_fields:
    x = pd.to_numeric(analysis[col], errors="coerce")
    rows.append({
        "field": col, "missing": int(x.isna().sum()),
        "zero": int(x.eq(0).sum()), "negative": int(x.lt(0).sum()),
        "positive": int(x.gt(0).sum())
    })
physical_review = pd.DataFrame(rows)
display(physical_review)

zero_insolation = analysis.loc[
    pd.to_numeric(analysis["pl_insol"], errors="coerce").eq(0),
    ["pl_name", "hostname", "pl_insol", "pl_eqt"]
]
print("pl_insol == 0 records:", len(zero_insolation))
display(zero_insolation.head(30))

**Interpretation:** zero-insolation values are flagged for investigation, not automatically deleted or declared invalid. Check NASA field definitions and the original source records before deciding what they mean.

## 6. Mass-provenance distribution

In [ ]:
mass_provenance = (
    analysis["pl_bmassprov"].fillna("Missing").astype(str)
    .value_counts(dropna=False).rename_axis("mass_provenance")
    .reset_index(name="planet_count")
)
mass_provenance["percentage"] = (
    100 * mass_provenance["planet_count"] / len(analysis)
).round(2)
display(mass_provenance)

## 7. Measurement-uncertainty availability and summaries

In [ ]:
uncertainty_rows = []
for col in uncertainty_cols:
    x = pd.to_numeric(analysis[col], errors="coerce")
    uncertainty_rows.append({
        "field": col, "available": int(x.notna().sum()),
        "missing": int(x.isna().sum()),
        "minimum": x.min(), "median": x.median(), "maximum": x.max()
    })
uncertainty_summary = pd.DataFrame(uncertainty_rows)
display(uncertainty_summary if not uncertainty_summary.empty else
        pd.DataFrame({"Note": ["No uncertainty columns were available."]}))

## 8. Statistical outlier flags (review only)
The IQR method identifies values far from the middle 50% of the observed data. Exoplanet populations can be broad and skewed, so these flags are not scientific invalidation.

In [ ]:
outlier_rows = []
for col in physical_fields:
    x = pd.to_numeric(analysis[col], errors="coerce").dropna()
    if len(x) < 4:
        continue
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    full_x = pd.to_numeric(analysis[col], errors="coerce")
    outlier_rows.append({
        "field": col, "q1": q1, "q3": q3, "iqr": iqr,
        "lower_review_bound": lo, "upper_review_bound": hi,
        "flagged_count": int((full_x.lt(lo) | full_x.gt(hi)).sum())
    })
outlier_summary = pd.DataFrame(outlier_rows)
display(outlier_summary)

## 9. Explore equilibrium temperature and insolation

In [ ]:
eqt = pd.to_numeric(analysis["pl_eqt"], errors="coerce").dropna()
insol = pd.to_numeric(analysis["pl_insol"], errors="coerce").dropna()
positive_insol = insol[insol > 0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(eqt, bins=40)
ax.set(title="Equilibrium temperature distribution", xlabel="pl_eqt (K)", ylabel="Planet count")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
if len(positive_insol):
    ax.hist(np.log10(positive_insol), bins=40)
    ax.set_xlabel("log10(pl_insol)")
    ax.set_ylabel("Planet count")
    ax.set_title("Positive incident-flux distribution")
else:
    ax.text(0.5, 0.5, "No positive insolation values", ha="center", va="center")
    ax.set_axis_off()
plt.tight_layout()
plt.show()

print(f"pl_eqt available: {eqt.notna().sum()} / {len(analysis)}")
print(f"positive pl_insol available: {len(positive_insol)} / {len(analysis)}")

## 10. Research-led habitable-zone screening protocol

### Scientific basis
This notebook uses the **conservative circumstellar habitable-zone (HZ) limits** from Kopparapu et al. (2014), for a 1-Earth-mass planet, with the published effective-temperature polynomial. The conservative inner edge is the runaway-greenhouse limit; the conservative outer edge is the maximum-greenhouse limit.

The polynomial is:

`S_eff = S_eff_sun + a*T' + b*T'^2 + c*T'^3 + d*T'^4`, where `T' = st_teff - 5780 K`.

The model is applied only within its stated stellar effective-temperature range of **2600–7200 K**. The optimistic limits (Recent Venus and Early Mars) are reported separately and are not treated as the conservative HZ.

### Interpretation safeguards
- `pl_insol` is the primary quantity compared with the star-specific HZ flux boundaries.
- `pl_eqt` is retained as complementary context; it is **not** assigned a universal temperature cutoff.
- HZ membership is not proof of surface liquid water or habitability.
- Kopparapu's model is intended for main-sequence stars. The available `st_spectype` field is used only as a screening clue; missing or ambiguous spectral classifications remain unverified.
- Zero or missing `pl_insol`, missing `st_teff`, or `st_teff` outside 2600–7200 K are not classified.
- The HZ calculation is a model-based research screen, not a confirmed planetary classification.

**Reference:** Kopparapu, R. K. et al. (2014), *Habitable Zones Around Main-sequence Stars: Dependence on Planetary Mass*, The Astrophysical Journal Letters, 787, L29. DOI: https://doi.org/10.1088/2041-8205/787/2/L29.


In [ ]:
# Kopparapu et al. (2014), Table 1: 1 Earth-mass planet coefficients.
# Coefficient order: (S_eff_sun, a, b, c, d)
KOPPARAPU_COEFFICIENTS = {
    "runaway_greenhouse": (1.107, 1.332e-4, 1.580e-8, -8.308e-12, -1.931e-15),
    "maximum_greenhouse": (0.356, 6.171e-5, 1.698e-9, -3.198e-12, -5.575e-16),
    "recent_venus": (1.776, 2.136e-4, 2.533e-8, -1.332e-11, -3.097e-15),
    "early_mars": (0.320, 5.547e-5, 1.526e-9, -2.874e-12, -5.011e-16),
}
KOPPARAPU_REFERENCE_T_K = 5780.0
KOPPARAPU_VALID_TEFF_MIN_K = 2600.0
KOPPARAPU_VALID_TEFF_MAX_K = 7200.0


def kopparapu_effective_flux(teff_k, boundary):
    """Return the stellar flux boundary (Earth flux units) for a given Teff."""
    s0, a, b, c, d = KOPPARAPU_COEFFICIENTS[boundary]
    delta_t = teff_k - KOPPARAPU_REFERENCE_T_K
    return s0 + a * delta_t + b * delta_t**2 + c * delta_t**3 + d * delta_t**4


teff = pd.to_numeric(analysis["st_teff"], errors="coerce")
model_applicable = teff.between(
    KOPPARAPU_VALID_TEFF_MIN_K, KOPPARAPU_VALID_TEFF_MAX_K, inclusive="both"
)

for boundary in KOPPARAPU_COEFFICIENTS:
    analysis[f"hz_{boundary}_flux"] = np.where(
        model_applicable,
        kopparapu_effective_flux(teff, boundary),
        np.nan
    )

# The flux boundaries are ordered from high incident flux (inner edge)
# to low incident flux (outer edge).
analysis["hz_model_teff_applicable"] = model_applicable
print("Host stars within Kopparapu Teff range:", int(model_applicable.sum()))
print("Host stars outside / missing Teff range:", int((~model_applicable).sum()))
display(analysis[["pl_name", "st_teff", "hz_runaway_greenhouse_flux",
                  "hz_maximum_greenhouse_flux", "hz_model_teff_applicable"]].head(10))


## 11. Build the candidate evidence table

In [ ]:
keep = [
    c for c in ["pl_name", "hostname", "disc_year", "discoverymethod",
                "pl_eqt", "pl_insol", "pl_bmasse", "pl_bmassprov", "sy_dist",
                "st_teff", "st_spectype", "pl_insolerr1", "pl_insolerr2",
                "hz_runaway_greenhouse_flux", "hz_maximum_greenhouse_flux",
                "hz_recent_venus_flux", "hz_early_mars_flux",
                "hz_model_teff_applicable"]
    if c in analysis.columns
]
screening = analysis[keep].copy()
eqt = pd.to_numeric(screening["pl_eqt"], errors="coerce")
ins = pd.to_numeric(screening["pl_insol"], errors="coerce")
inner = pd.to_numeric(screening["hz_runaway_greenhouse_flux"], errors="coerce")
outer = pd.to_numeric(screening["hz_maximum_greenhouse_flux"], errors="coerce")
optimistic_inner = pd.to_numeric(screening["hz_recent_venus_flux"], errors="coerce")
optimistic_outer = pd.to_numeric(screening["hz_early_mars_flux"], errors="coerce")

screening["eqt_available"] = eqt.notna()
screening["insol_available"] = ins.notna()
screening["thermal_evidence_status"] = np.select(
    [eqt.notna() & ins.notna(), eqt.notna() & ins.isna(),
     eqt.isna() & ins.notna()],
    ["Both indicators available", "Equilibrium temperature only", "Insolation only"],
    default="Neither indicator available"
)

# Spectral type is only a screening clue. It does not independently prove
# main-sequence status. Explicit luminosity classes III/IV/II/I are flagged.
sptype = screening.get("st_spectype", pd.Series(index=screening.index, dtype="object")).fillna("").astype(str).str.upper().str.strip()
screening["stellar_class_evidence"] = np.select(
    [sptype.str.contains(r"\bIII\b|\bIV\b|\bII\b|\bI\b", regex=True),
     sptype.str.contains(r"\bV\b", regex=True)],
    ["Evolved-star luminosity class indicated", "Class V indicated"],
    default="Unverified / spectral type unavailable"
)

teff_ok = screening["hz_model_teff_applicable"].fillna(False).astype(bool)
valid_flux = ins.notna() & ins.gt(0)
conservative = valid_flux & teff_ok & ins.le(inner) & ins.ge(outer)
optimistic = valid_flux & teff_ok & ins.le(optimistic_inner) & ins.ge(optimistic_outer)

screening["hz_screening_category"] = "Not classified — insufficient or out-of-range evidence"
screening.loc[valid_flux & ~teff_ok, "hz_screening_category"] =     "Not classified — host Teff outside model range or missing"
screening.loc[ins.eq(0), "hz_screening_category"] =     "Not classified — zero insolation requires source review"
screening.loc[valid_flux & teff_ok & ~optimistic, "hz_screening_category"] =     "Outside optimistic HZ flux range"
screening.loc[optimistic & ~conservative, "hz_screening_category"] =     "Within optimistic HZ only"
screening.loc[conservative, "hz_screening_category"] =     "Within conservative HZ flux range — stellar class unverified"
screening.loc[conservative & screening["stellar_class_evidence"].eq("Class V indicated"),
               "hz_screening_category"] = "Conservative HZ candidate — Class V indicated"
screening.loc[conservative & screening["stellar_class_evidence"].eq("Evolved-star luminosity class indicated"),
               "hz_screening_category"] = "Review — evolved-star class indicated"

# Carry signed NASA asymmetric errors into a flux interval where both are usable.
err_low = pd.to_numeric(screening.get("pl_insolerr1"), errors="coerce")
err_high = pd.to_numeric(screening.get("pl_insolerr2"), errors="coerce")
uncertainty_available = err_low.notna() & err_high.notna() & ~(err_low.eq(0) & err_high.eq(0))
screening["insol_uncertainty_available"] = uncertainty_available
screening["insol_interval_lower"] = np.where(uncertainty_available, np.maximum(0, ins + np.minimum(err_low, err_high)), np.nan)
screening["insol_interval_upper"] = np.where(uncertainty_available, ins + np.maximum(err_low, err_high), np.nan)

lower = pd.to_numeric(screening["insol_interval_lower"], errors="coerce")
upper = pd.to_numeric(screening["insol_interval_upper"], errors="coerce")
robust_inside = uncertainty_available & teff_ok & (lower >= outer) & (upper <= inner)
uncertainty_overlap = uncertainty_available & teff_ok & (upper >= outer) & (lower <= inner) & ~robust_inside
screening["insol_uncertainty_assessment"] = "Uncertainty unavailable or zero-valued in archive"
screening.loc[uncertainty_available, "insol_uncertainty_assessment"] = "Interval assessed"
screening.loc[robust_inside, "insol_uncertainty_assessment"] = "Entire reported interval within conservative HZ"
screening.loc[uncertainty_overlap, "insol_uncertainty_assessment"] = "Uncertainty interval overlaps conservative HZ boundary"
screening.loc[uncertainty_available & teff_ok & ~uncertainty_overlap & ~robust_inside,
               "insol_uncertainty_assessment"] = "Reported interval outside conservative HZ"
screening.loc[uncertainty_available & ~teff_ok, "insol_uncertainty_assessment"] = \
    "Not assessed — host Teff outside model range or missing"

# Candidate target for later ML work. This is a protocol-derived pseudo-label,
# not an independently observed habitability ground truth.
screening["ml_target_conservative_hz"] = np.nan
eligible_ml = valid_flux & teff_ok & screening["stellar_class_evidence"].eq("Class V indicated")
screening.loc[eligible_ml, "ml_target_conservative_hz"] = conservative.loc[eligible_ml].astype(int)
screening["ml_target_note"] = "Derived from Kopparapu HZ screening; not a habitability ground-truth label"

display(screening["hz_screening_category"].value_counts(dropna=False).to_frame("planet_count"))
display(screening["insol_uncertainty_assessment"].value_counts(dropna=False).to_frame("planet_count"))
display(screening.loc[screening["hz_screening_category"].str.contains("candidate|Within conservative", case=False),
                      ["pl_name", "hostname", "st_teff", "st_spectype", "pl_insol",
                       "hz_runaway_greenhouse_flux", "hz_maximum_greenhouse_flux",
                       "pl_eqt", "insol_uncertainty_assessment", "hz_screening_category"]].head(30))


## 12. Validate and export Milestone 4 reports

In [ ]:
validation = {
    "input_rows": int(len(m3)),
    "screening_rows": int(len(screening)),
    "rows_removed": int(len(m3) - len(screening)),
    "missing_pl_eqt": int(pd.to_numeric(analysis["pl_eqt"], errors="coerce").isna().sum()),
    "missing_pl_insol": int(pd.to_numeric(analysis["pl_insol"], errors="coerce").isna().sum()),
    "zero_insolation_flagged": int(pd.to_numeric(analysis["pl_insol"], errors="coerce").eq(0).sum()),
    "kopparapu_model": "Kopparapu et al. (2014), 1 Earth-mass coefficients",
    "teff_model_range_K": [KOPPARAPU_VALID_TEFF_MIN_K, KOPPARAPU_VALID_TEFF_MAX_K],
    "raw_file_overwritten": False,
    "milestone3_file_overwritten": False,
    "habitability_claim": False,
    "ml_target_is_protocol_derived_pseudolabel": True
}
display(pd.DataFrame([validation]))
assert validation["rows_removed"] == 0
assert screening["pl_name"].is_unique
assert len(screening) == len(m3)
print("Milestone 4 structural validation passed. Scientific review outputs must still be interpreted.")

OUT = PROJECT_ROOT / "DATA/processed/milestone4"
OUT.mkdir(parents=True, exist_ok=True)

physical_review.to_csv(OUT / "m4_physical_value_review.csv", index=False)
zero_insolation.to_csv(OUT / "m4_zero_insolation_records.csv", index=False)
mass_provenance.to_csv(OUT / "m4_mass_provenance_summary.csv", index=False)
uncertainty_summary.to_csv(OUT / "m4_uncertainty_summary.csv", index=False)
outlier_summary.to_csv(OUT / "m4_statistical_outlier_flags.csv", index=False)
screening.to_csv(OUT / "m4_habitable_zone_screening.csv", index=False)
screening.loc[screening["ml_target_conservative_hz"].notna()].to_csv(
    OUT / "m4_ml_training_labels_review.csv", index=False
)

with open(OUT / "m4_screening_protocol.json", "w", encoding="utf-8") as f:
    json.dump({
        "source": "Kopparapu et al. (2014), ApJL 787, L29",
        "doi": "10.1088/2041-8205/787/2/L29",
        "model": "1 Earth-mass; runaway greenhouse inner edge; maximum greenhouse outer edge",
        "reference_teff_K": KOPPARAPU_REFERENCE_T_K,
        "valid_teff_range_K": [KOPPARAPU_VALID_TEFF_MIN_K, KOPPARAPU_VALID_TEFF_MAX_K],
        "coefficients": {k: list(v) for k, v in KOPPARAPU_COEFFICIENTS.items()},
        "primary_screening_variable": "pl_insol (Earth incident-flux units)",
        "complementary_variable": "pl_eqt (no universal cutoff applied)",
        "limitations": [
            "HZ membership is not proof of habitability or life",
            "main-sequence status is only screened using available spectral-type metadata",
            "zero and missing insolation are not classified",
            "ML target is a protocol-derived pseudo-label, not independent ground truth"
        ]
    }, f, indent=2)
with open(OUT / "m4_validation_summary.json", "w", encoding="utf-8") as f:
    json.dump(validation, f, indent=2)

print("Reports saved to:", OUT)
for report_path in sorted(OUT.iterdir()):
    print("-", report_path.name)


## 16. Automated Milestone 4 results summary

In [ ]:
summary_rows = [
    {"metric": "Input planets", "value": len(m3)},
    {"metric": "Rows retained", "value": len(screening)},
    {"metric": "Zero-insolation records flagged", "value": int(pd.to_numeric(analysis["pl_insol"], errors="coerce").eq(0).sum())},
    {"metric": "Conservative HZ candidates (Class V indicated)", "value": int(screening["hz_screening_category"].eq("Conservative HZ candidate — Class V indicated").sum())},
    {"metric": "Conservative HZ flux range (stellar class unverified)", "value": int(screening["hz_screening_category"].eq("Within conservative HZ flux range — stellar class unverified").sum())},
    {"metric": "Optimistic HZ only", "value": int(screening["hz_screening_category"].eq("Within optimistic HZ only").sum())},
    {"metric": "ML pseudo-labels available", "value": int(screening["ml_target_conservative_hz"].notna().sum())},
    {"metric": "ML positive labels", "value": int(screening["ml_target_conservative_hz"].eq(1).sum())},
    {"metric": "ML negative labels", "value": int(screening["ml_target_conservative_hz"].eq(0).sum())},
    {"metric": "Uncertainty intervals fully within conservative HZ", "value": int(screening["insol_uncertainty_assessment"].eq("Entire reported interval within conservative HZ").sum())},
    {"metric": "Uncertainty intervals overlapping HZ boundary", "value": int(screening["insol_uncertainty_assessment"].eq("Uncertainty interval overlaps conservative HZ boundary").sum())},
]
m4_results_summary = pd.DataFrame(summary_rows)
display(m4_results_summary)
m4_results_summary.to_csv(OUT / "m4_results_summary.csv", index=False)


## 13. Milestone 4 conclusion and review checklist

1. **Data integrity:** confirm all 6,451 input records remain; no rows were deleted.
2. **Scientific protocol:** document the Kopparapu et al. (2014) 1-Earth-mass conservative HZ model and its 2600–7200 K applicability range.
3. **Zero insolation:** retain all zero-flux records as unclassified pending interpretation of source flags and provenance; do not interpret zero as physically zero received energy without evidence.
4. **Mass provenance:** report the archive's mass-provenance distribution and retain the distinction between measured mass and Msini / mass-radius estimates.
5. **Uncertainty:** distinguish missing/zero-reported uncertainty from an assessed uncertainty interval. Review candidate intervals near HZ boundaries.
6. **Outliers:** treat IQR flags as review indicators, not invalid records. Use astrophysical context before making exclusions.
7. **Thermal evidence:** report availability of `pl_eqt` and `pl_insol`; equilibrium temperature is complementary evidence, not a universal threshold.
8. **Stellar applicability:** inspect spectral-type evidence and the model temperature range; unknown or evolved stellar classes require caution.
9. **ML handover:** `ml_target_conservative_hz` is a protocol-derived pseudo-label for later model experimentation. It is not independent habitability ground truth. Avoid target leakage in Milestone 5.
10. **Limitation:** potentially inside a theoretical HZ does not mean habitable. Atmosphere, clouds, albedo, water inventory, magnetic environment, and geological conditions are not established by this catalogue alone.

**Milestone 4 completion requires:** successful execution, report review, documented treatment of flagged records, and a written interpretation of the resulting candidate list. A successful code assertion alone is not scientific completion.


**Final sign-off:** complete this checklist after running the notebook against the actual project-folder M3 and NASA source files, then inspect `m4_results_summary.csv` and the candidate review queue.

## 14. Review zero-insolation records against NASA source flags

The 46 zero values were retained in Milestone 3/4. This section checks whether the archive's limit flag and uncertainty fields indicate that zero is a placeholder/limit rather than a measured physical flux. Do not silently replace the reported value.

In [ ]:
zero_review_cols = [c for c in ["pl_name", "hostname", "pl_insol", "pl_insolerr1", "pl_insolerr2", "pl_insollim", "pl_eqt", "st_teff", "st_spectype"] if c in analysis.columns]
zero_insolation_diagnostic = analysis.loc[pd.to_numeric(analysis["pl_insol"], errors="coerce").eq(0), zero_review_cols].copy()
if "pl_insollim" in zero_insolation_diagnostic.columns:
    zero_insolation_diagnostic["archive_limit_flag_interpretation"] = zero_insolation_diagnostic["pl_insollim"].map({0: "Archive flag 0 — consult current NASA column documentation", 1: "Archive flag 1 — consult current NASA column documentation"}).fillna("Missing / other flag")
print("Zero-insolation records retained for manual source review:", len(zero_insolation_diagnostic))
if "pl_insollim" in zero_insolation_diagnostic.columns:
    display(zero_insolation_diagnostic["pl_insollim"].value_counts(dropna=False).to_frame("record_count"))
display(zero_insolation_diagnostic.head(50))
zero_insolation_diagnostic.to_csv(OUT / "m4_zero_insolation_source_diagnostic.csv", index=False)


### Dataset-specific zero-insolation finding

In the supplied NASA composite file, all **46** records with `pl_insol = 0` have `discoverymethod = Imaging` and `pl_insollim = 0`. For 27 records, both reported `pl_insol` uncertainty values are zero; for the remaining 19, at least one uncertainty value is missing. These records do not provide a usable positive incident-flux measurement for the HZ calculation. They are therefore retained but excluded from HZ classification and the ML target. This is an interpretation of the archive fields in this snapshot, not a claim that the planets physically receive zero stellar energy.


## 15. Prioritise uncertainty and outlier review for HZ candidates

This section creates a focused review queue. It does not remove records or declare any candidate habitable. Candidate rows with unavailable uncertainties are explicitly retained with that limitation.

In [ ]:
review_queue = screening.loc[
    screening["hz_screening_category"].str.contains("candidate|Within conservative|Within optimistic", case=False, na=False)
].copy()
review_queue["needs_uncertainty_followup"] = ~review_queue["insol_uncertainty_available"]
review_queue["needs_stellar_class_followup"] = review_queue["stellar_class_evidence"].eq("Unverified / spectral type unavailable")
review_queue["review_priority"] = np.select(
    [review_queue["insol_uncertainty_assessment"].str.contains("overlaps", case=False, na=False),
     review_queue["needs_uncertainty_followup"],
     review_queue["needs_stellar_class_followup"]],
    ["High — HZ boundary overlap", "Medium — uncertainty unavailable", "Medium — stellar class unverified"],
    default="Standard — inspect candidate evidence"
)
print("Candidate review queue:", len(review_queue))
display(review_queue["review_priority"].value_counts().to_frame("planet_count"))
display(review_queue.head(50))
review_queue.to_csv(OUT / "m4_candidate_review_queue.csv", index=False)
